# SIC8 — 08A v3.2 | AWS S3 Bronze → Databricks Auto Loader: Reliable Continuous Ingestion

**Continuous Streaming Ingestion (24/7).** This notebook discovers new S3  objects from  and writes them continuously to the Delta Raw table .

* **v3.2 Update:** Added  with  alongside .
* Preserves exact Raw Delta contracts and preflight gates.


In [0]:
# Edit RUN_STAGE and STREAMING_MODE as needed.
RUN_STAGE = "INGEST"            # PREFLIGHT | INGEST
STREAMING_MODE = "CONTINUOUS"   # CONTINUOUS | AVAILABLE_NOW
TRIGGER_INTERVAL = "10 seconds"
CATALOG = "workspace"
SCHEMA = "sic8_streaming"
SOURCE_S3 = "s3://sic-pipeline-storage/bronze/transactions/"
STATE_VOLUME = f"/Volumes/{CATALOG}/{SCHEMA}/streaming_state"
STATE_ROOT = f"{STATE_VOLUME}/autoloader_tx_raw_v3"  # Stable volume root
CHECKPOINT_PATH = f"{STATE_ROOT}/checkpoint"
SCHEMA_PATH = f"{STATE_ROOT}/schema"
RAW_TABLE = f"{CATALOG}.{SCHEMA}.streaming_raw_envelopes_v3"
MAX_FILES_PER_TRIGGER = 500

assert RUN_STAGE in ("PREFLIGHT", "INGEST")
assert STREAMING_MODE in ("CONTINUOUS", "AVAILABLE_NOW")
assert SOURCE_S3 == "s3://sic-pipeline-storage/bronze/transactions/"
assert STATE_ROOT.startswith(f"/Volumes/{CATALOG}/{SCHEMA}/")
print("RUN_STAGE:", RUN_STAGE)
print("STREAMING_MODE:", STREAMING_MODE, f"(interval={TRIGGER_INTERVAL})")
print("SOURCE:", SOURCE_S3)
print("DELTA RAW:", RAW_TABLE)
print("CHECKPOINT:", CHECKPOINT_PATH)


RUN_STAGE: INGEST
SOURCE: s3://sic-pipeline-storage/bronze/transactions/
DELTA RAW: workspace.sic8_streaming.streaming_raw_envelopes_v3
CHECKPOINT: /Volumes/workspace/sic8_streaming/streaming_state/autoloader_tx_raw_v3/checkpoint


## Gate 1 — AWS source access and actual files (read-only)

`Storage Credential → External Location → READ FILES` must cover the **Bronze prefix**, not only the historical `raw/baseline/` prefix. Credentials allow `List/Get`; they **do not** refresh a batch DataFrame or trigger ingestion. This code does not create IAM roles and does not expose keys.

In [0]:
from pyspark.sql import functions as F, types as T
from datetime import datetime, timezone
import json

spark.conf.set("spark.sql.session.timeZone", "UTC")
source_ok = False
sample_paths = []
try:
    root_objects = dbutils.fs.ls(SOURCE_S3)
    directories = sorted([x.path for x in root_objects if x.isDir()])
    files = [x for x in root_objects if not x.isDir() and x.path.endswith(".json.gz")]
    for folder in directories:
        files.extend(x for x in dbutils.fs.ls(folder)
                     if (not x.isDir()) and x.path.endswith(".json.gz"))
    files = sorted(files, key=lambda x: (x.modificationTime, x.path), reverse=True)
    sample_paths = [x.path for x in files[:3]]
    source_ok = len(sample_paths) > 0
    print("S3_SUBDIRECTORIES:", len(directories))
    print("S3_GZIP_OBJECTS_LISTED:", len(files))
    print("LATEST_S3_FILES:", [
        {"file": x.path.rsplit("/", 1)[-1],
         "last_modified_utc": datetime.fromtimestamp(x.modificationTime/1000, timezone.utc).isoformat()}
        for x in files[:3]])
    print("S3_READ_GATE:", "PASS" if source_ok else "FAIL - no *.json.gz objects")
except Exception as exc:
    print("S3_READ_GATE: FAIL")
    print("ERROR_TYPE:", type(exc).__name__)
    print("MESSAGE:", str(exc)[:1100])
    print("CHECK: external location covering S3 bronze/, storage credential IAM role, READ FILES, GetObject/ListBucket, KMS if necessary")

S3_SUBDIRECTORIES: 4
S3_GZIP_OBJECTS_LISTED: 5558
LATEST_S3_FILES: [{'file': '00000000000000167928.json.gz', 'last_modified_utc': '2026-10-29T14:20:25+00:00'}, {'file': '00000000000000110045.json.gz', 'last_modified_utc': '2026-10-29T14:20:24+00:00'}, {'file': '00000000000000086195.json.gz', 'last_modified_utc': '2026-10-29T14:20:24+00:00'}]
S3_READ_GATE: PASS


## Gate 2 — Inspect real S3 gzip samples and parser tolerance (read-only)

The ingestion stage is a **raw landing**, not transaction-only validation. Never reject the whole source because one event has a different `event_type`, `schema_version` (for example `"1"` instead of `"1.0.0"`), or optional `data.transaction_id`. Keep every line; flag schema mismatches for a later Silver/Quarantine stage. Only a structurally unreadable input or completely unparseable source should stop ingestion.


In [0]:
# Parse BOTH forms of the S3 Kafka wrapper: value as JSON object or JSON-encoded string.
# No rows are dropped or changed in Raw; this expression is for diagnostic metadata only.
def event_json(frame):
    return frame.withColumn("_event_json", F.get_json_object(F.col("raw_json_line"), "$.value"))

sample_ok = False
try:
    if not source_ok:
        raise RuntimeError("S3 read gate failed; cannot test sample")
    sample_df = (spark.read.format("text")
        .load(sample_paths)
        .select(F.col("value").alias("raw_json_line")))
    examined = (event_json(sample_df).select(
        "raw_json_line", "_event_json",
        F.get_json_object("_event_json", "$.event_id").alias("event_id"),
        F.get_json_object("_event_json", "$.event_type").alias("event_type"),
        F.get_json_object("_event_json", "$.schema_version").alias("schema_version"),
        F.get_json_object("_event_json", "$.data.transaction_id").alias("transaction_id"),
    ).limit(100).collect())
    if not examined:
        raise RuntimeError("Selected S3 gzip files contain no JSONL records")
    valid_events = [r for r in examined if r._event_json is not None and r.event_id]
    sample_ok = len(valid_events) > 0
    print("REAL_SAMPLE_ROWS:", len(examined))
    print("PARSEABLE_WITH_EVENT_ID:", len(valid_events))
    print("MISSING_EVENT_ID_OR_UNPARSEABLE:", len(examined) - len(valid_events))
    print("EVENT_TYPES:", sorted({str(r.event_type) for r in examined}))
    print("SCHEMA_VERSIONS:", sorted({str(r.schema_version) for r in examined}))
    print("MISSING_TRANSACTION_ID:", sum(not r.transaction_id for r in examined))
    print("REAL_EVENT_IDS (sample):", [r.event_id for r in valid_events[:4]])
    print("REAL_GZIP_PARSER_GATE:", "PASS" if sample_ok else "FAIL - no parseable event with event_id")
    if len(valid_events) != len(examined):
        print("WARNING: Some raw events need downstream quarantine/review. Raw landing will retain ALL lines.")
    if any(r.event_type != "transaction.created" for r in examined):
        print("INFO: Event types differ from historical fixture; NOT an ingestion error.")
except Exception as exc:
    sample_ok = False
    print("REAL_GZIP_PARSER_GATE: FAIL", type(exc).__name__, str(exc)[:900])

# Deterministic fixture: value as an object and as an escaped JSON string.
# Also demonstrate that differing schema versions are harmless for Raw ingestion.
e = {"schema_version":"1.0.0", "event_type":"transaction.created", "event_id":"fixture-0001", "data":{"transaction_id":"fixture-tx"}}
e2 = {"schema_version":"1", "event_type":"payment.created", "event_id":"fixture-0002", "data":{"amount":58.75}}
fixture_lines = [json.dumps({"offset":1,"value":e}), json.dumps({"offset":2,"value":json.dumps(e)}), json.dumps({"offset":3,"value":e2})]
fx = spark.createDataFrame([(s,) for s in fixture_lines], "raw_json_line STRING")
fx_ids = [r.event_id for r in event_json(fx).select(F.get_json_object("_event_json", "$.event_id").alias("event_id")).collect()]
assert fx_ids == ["fixture-0001", "fixture-0001", "fixture-0002"], fx_ids
print("FIXTURE_PARSER_GATE: PASS (object, encoded-string, schema variant)")


REAL_SAMPLE_ROWS: 100
PARSEABLE_WITH_EVENT_ID: 100
MISSING_EVENT_ID_OR_UNPARSEABLE: 0
EVENT_TYPES: ['transaction.created']
SCHEMA_VERSIONS: ['1']
MISSING_TRANSACTION_ID: 100
REAL_EVENT_IDS (sample): ['9e935c3f-cc31-4923-b4ba-2d1804c19ae4', '08be498f-c976-46eb-af4c-659b24495c66', '1a359144-078b-476d-98a2-02086873d735', '6de27e23-8124-46a3-8e1b-b4981153b2d1']
REAL_GZIP_PARSER_GATE: PASS
FIXTURE_PARSER_GATE: PASS (object, encoded-string, schema variant)


## Gate 3 — Unity Catalog destination readiness

Preflight checks schema/volume without creating anything. **INGEST** can create an ordinary managed schema and managed volume if your principal has privileges; otherwise the workspace admin must provide them. The checkpoint is **not stored in S3 Bronze**. Don't delete/recreate checkpoints to force data freshness.

In [0]:
try:
    schemas = [r.databaseName for r in spark.sql(f"SHOW SCHEMAS IN `{CATALOG}`").collect()]
    schema_exists = SCHEMA in schemas
    print("UC_SCHEMA:", "EXISTS" if schema_exists else "MISSING — INGEST will attempt CREATE SCHEMA")
except Exception as exc:
    schema_exists = False
    print("UC_SCHEMA_DIAGNOSTIC:", type(exc).__name__, str(exc)[:650])

try:
    dbutils.fs.ls(STATE_VOLUME)
    print("UC_MANAGED_VOLUME: EXISTS", STATE_VOLUME)
except Exception as exc:
    print("UC_MANAGED_VOLUME: NOT READY — INGEST will attempt CREATE VOLUME")
    print("VOLUME_MESSAGE:", str(exc)[:250])

preflight_ok = bool(source_ok and sample_ok)  # Raw gate; event-type/transaction-id checks deferred to Silver
print("SOURCE_AND_PARSER_PREFLIGHT:", "PASS" if preflight_ok else "FAIL")
if RUN_STAGE == "INGEST" and not preflight_ok:
    raise RuntimeError("STOP: No streaming writes until S3 read and parser gates PASS")

UC_SCHEMA: EXISTS
UC_MANAGED_VOLUME: EXISTS /Volumes/workspace/sic8_streaming/streaming_state
SOURCE_AND_PARSER_PREFLIGHT: PASS


## Stage A — One bounded Auto Loader run (`RUN_STAGE="INGEST"`)

* Uses **directory listing**, no SQS permissions or SNS notifications.
* Explicitly disables the deprecated **incremental lexical listing**: parallel Kafka partition folders can grow at different rates.
* `includeExistingFiles=true` backfills files **only at the first run for this checkpoint**.
* `availableNow=True` stops after processing current files; run again via a Job, using the **same checkpoint** to discover new ones.
* Appends original JSONL lines into a **UC managed Delta Raw table**, not S3 Bronze. One new source file can contain multiple lines.
* Raw can contain simulator truth; do not make it accessible to the model's feature pipeline.

In [0]:
if RUN_STAGE == "INGEST":
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS .")
    spark.sql(f"CREATE VOLUME IF NOT EXISTS ..")
    dbutils.fs.ls(STATE_VOLUME)

    spark.sql(f"""CREATE TABLE IF NOT EXISTS .. (
        raw_json_line STRING,
        source_file STRING,
        source_modified_utc TIMESTAMP,
        source_size_bytes BIGINT,
        event_id STRING,
        event_type STRING,
        schema_version STRING,
        kafka_offset BIGINT,
        kafka_key STRING,
        ingested_at_utc TIMESTAMP
    ) USING DELTA""")

    def path_exists(path):
        try:
            dbutils.fs.ls(path)
            return True
        except Exception:
            return False

    has_checkpoint = path_exists(CHECKPOINT_PATH)
    previous_count = spark.table(RAW_TABLE).count()
    if previous_count > 0 and not has_checkpoint:
        raise RuntimeError("RAW table already has rows but checkpoint is absent. STOP to prevent re-ingestion; do not reset state.")
    if previous_count == 0 and has_checkpoint:
        raise RuntimeError("Checkpoint exists while Raw table has zero rows. STOP to inspect orphaned state before writing.")

    stream = (spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "text")
        .option("cloudFiles.schemaLocation", SCHEMA_PATH)
        .option("cloudFiles.useNotifications", "false")
        .option("cloudFiles.useIncrementalListing", "false")
        .option("cloudFiles.includeExistingFiles", "true")
        .option("cloudFiles.allowOverwrites", "false")
        .option("cloudFiles.maxFilesPerTrigger", str(MAX_FILES_PER_TRIGGER))
        .option("pathGlobFilter", "*.json.gz")
        .load(SOURCE_S3))
    raw = (stream.select(
        F.col("value").alias("raw_json_line"),
        F.col("_metadata.file_path").alias("source_file"),
        F.col("_metadata.file_modification_time").alias("source_modified_utc"),
        F.col("_metadata.file_size").cast("bigint").alias("source_size_bytes"),
        F.get_json_object(F.get_json_object(F.col("value"), "$.value"), "$.event_id").alias("event_id"),
        F.get_json_object(F.get_json_object(F.col("value"), "$.value"), "$.event_type").alias("event_type"),
        F.get_json_object(F.get_json_object(F.col("value"), "$.value"), "$.schema_version").alias("schema_version"),
        F.expr("try_cast(get_json_object(value, '$.offset') AS BIGINT)").alias("kafka_offset"),
        F.get_json_object(F.col("value"), "$.key").alias("kafka_key"),
        F.current_timestamp().alias("ingested_at_utc")))

    print("RAW_BEFORE_ROWS:", previous_count)
    trigger_arg = {"processingTime": TRIGGER_INTERVAL} if STREAMING_MODE == "CONTINUOUS" else {"availableNow": True}
    query = (raw.writeStream.format("delta")
        .option("checkpointLocation", CHECKPOINT_PATH)
        .trigger(**trigger_arg)
        .toTable(RAW_TABLE))
    print(f"RAW_STREAM_RUNNING: mode={STREAMING_MODE}, trigger={trigger_arg}")
    if STREAMING_MODE == "AVAILABLE_NOW":
        query.awaitTermination()
        if query.exception() is not None:
            raise RuntimeError("Auto Loader failed: " + str(query.exception()))
        print("RAW_STREAM_COMPLETED; lastProgress:", query.lastProgress)
        after_count = spark.table(RAW_TABLE).count()
        print("RAW_AFTER_ROWS:", after_count, "NEW_ROWS_THIS_RUN:", after_count - previous_count)
    else:
        print("Continuous stream active. Awaiting termination...")
        query.awaitTermination()
else:
    print("PREFLIGHT: no schema/table/checkpoint creation, no streaming write.")


RAW_BEFORE_ROWS: 0


26/10/29 14:22:08 Spark Server has not sent updates for Streaming Query b6e62439-f34a-46c7-811f-a6eae62fa0a0 in 60 seconds, but the query is still active. Marking query as in-progress. Spark Session ID is d15073dc-aeac-4891-a077-4eb31142577c. This is typically not a problem.
26/10/29 14:22:34 Spark Server has not sent updates for Streaming Query b6e62439-f34a-46c7-811f-a6eae62fa0a0 in 60 seconds, but the query is still active. Marking query as in-progress. Spark Session ID is d15073dc-aeac-4891-a077-4eb31142577c. This is typically not a problem.
26/10/29 14:22:59 Spark Server has not sent updates for Streaming Query b6e62439-f34a-46c7-811f-a6eae62fa0a0 in 60 seconds, but the query is still active. Marking query as in-progress. Spark Session ID is d15073dc-aeac-4891-a077-4eb31142577c. This is typically not a problem.
26/10/29 14:23:24 Spark Server has not sent updates for Streaming Query b6e62439-f34a-46c7-811f-a6eae62fa0a0 in 60 seconds, but the query is still active. Marking query as 

RAW_STREAM_COMPLETED; lastProgress: {
    "id": "26d30d3f-d253-4dc7-9989-36dc76d9fa31",
    "runId": "b6e62439-f34a-46c7-811f-a6eae62fa0a0",
    "name": null,
    "timestamp": "2026-10-29T14:26:32.881Z",
    "batchId": 11,
    "batchDuration": 165029,
    "durationMs": {
        "triggerExecution": 165029,
        "queryPlanning": 378,
        "collectSourceMetrics": 2,
        "getBatch": 15,
        "commitOffsets": 75,
        "postCommitBatchCallback": 5,
        "addBatch": 19267,
        "latestOffset": 26,
        "postAddBatchCallback": 409,
        "commitBatch": 548,
        "walCommit": 90
    },
    "eventTime": {},
    "stateOperators": [],
    "sources": [
        {
            "description": "CloudFilesSource[s3://sic-pipeline-storage/bronze/transactions/]",
            "startOffset": "{\"seqNum\":6001,\"sourceVersion\":3,\"lastBackfillStartTimeMs\":1793283655426,\"lastBackfillFinishTimeMs\":1793283824112,\"lastInputPath\":\"s3://sic-pipeline-storage/bronze/transactions/

## Verification — repeat this notebook after Sayed's connector writes another S3 file

A successful **freshness proof** is: first INGEST run → record metrics → new transaction produced via Kafka → S3 sink flushes new `.json.gz` file → repeat INGEST **with the same checkpoint** → Raw rows and event IDs increase. The ingester does not require new IAM privileges to notice these files.

In [0]:
if spark.catalog.tableExists(RAW_TABLE):
    r = spark.table(RAW_TABLE)
    m = r.agg(
        F.count("*").alias("raw_lines"),
        F.countDistinct("source_file").alias("files_ingested"),
        F.countDistinct("event_id").alias("distinct_event_ids"),
        F.sum(F.when(F.col("event_id").isNull(), 1).otherwise(0)).alias("missing_event_ids"),
        F.max("source_modified_utc").alias("latest_source_file_utc"),
        F.max("ingested_at_utc").alias("latest_ingest_utc")
    ).first().asDict()
    print("RAW_DELTA_VERIFICATION:", m)
    print("NOTE: raw_lines may exceed distinct_event_ids because Kafka can replay events; event dedup belongs in Silver.")
else:
    print("RAW TABLE NOT CREATED YET — expected during PREFLIGHT.")
print("NEXT: schedule this notebook as an INGEST job; availableNow is a finite run, not a continuously running service.")

RAW_DELTA_VERIFICATION: {'raw_lines': 275151, 'files_ingested': 5558, 'distinct_event_ids': 275151, 'missing_event_ids': 0, 'latest_source_file_utc': datetime.datetime(2026, 10, 29, 14, 20, 25), 'latest_ingest_utc': datetime.datetime(2026, 10, 29, 14, 26, 32, 908000)}
NOTE: raw_lines may exceed distinct_event_ids because Kafka can replay events; event dedup belongs in Silver.
NEXT: schedule this notebook as an INGEST job; availableNow is a finite run, not a continuously running service.


## If the next file is NOT detected

1. `dbutils.fs.ls(SOURCE_S3)` fails → **IAM role / UC storage credential / External Location / READ FILES**, not checkpoint.
2. `dbutils.fs.ls` sees a new object but Raw doesn't → **Was INGEST rerun?** Is the **same checkpoint** used? Is `*.json.gz` filter correct? Is the job running successfully?
3. Source has no new object → ask Sayed whether Kafka S3 sink has **flushed** a new gzip object. A Kafka event alone does not immediately imply a new S3 file.
4. Raw updates but Silver does not → downstream **Silver processing**, not AWS authorization.
5. `availableNow` is a bounded stream, so configure a **scheduled Job** for repeated discovery. Later, admin can enable UC **Managed File Events** (SQS/event notifications) and migrate with the **same checkpoint**, but directory listing is enough for the first reliable integration.

**Next engineering milestone (separate work):** derive stable event-level deduplicated Silver, quarantines, isolated simulator truth, and CDC. Do not claim that Raw landing alone delivers those datasets.